# Demo 06. Quadrature: Newton-Cotes, Composite Rules, and Adaptive Simpson

**Module 4** (quadrature), sections 4.1 and 4.4.

An interpolatory quadrature rule integrates the polynomial interpolant of the integrand. Fitting the interpolant on equispaced nodes gives the Newton-Cotes rules: midpoint, trapezoid, and Simpson. Applying a rule on many small panels gives a composite rule whose error decays at a fixed rate. When the integrand has a localized feature, a uniform panel width is wasteful; adaptive Simpson refines only where an error estimate demands it. This demo measures the composite convergence rates and shows the adaptive method concentrating work where it is needed.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider, IntSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Rules and their orders

On a panel of width $h$ the local errors are $O(h^3)$ for midpoint and trapezoid and $O(h^5)$ for Simpson. Summing over $n=(b-a)/h$ panels lowers each power by one, so the composite errors are

$$ E_{\text{trap}},\,E_{\text{mid}} = O(h^2), \qquad E_{\text{Simpson}} = O(h^4). $$

Simpson gains two extra orders because it integrates a quadratic through the panel and is in fact exact for cubics: its degree of exactness is three. On log-log axes the composite error against the number of panels is a straight line whose slope is the negative of the order.

In [ ]:
# ---------------------------------------------------------------------------
# Composite Newton-Cotes rules on [a, b] with n panels
# ---------------------------------------------------------------------------
# Each rule samples the integrand on a uniform grid and forms a weighted sum.
# The weights are exactly those that integrate the local interpolating
# polynomial: constants for midpoint, a line for trapezoid, a parabola for
# Simpson.

def composite_midpoint(f, a, b, n):
    """Composite midpoint rule. Samples panel centers. Order 2."""
    h = (b - a) / n
    x = a + (np.arange(n) + 0.5) * h        # panel midpoints
    return h * np.sum(f(x))

def composite_trapezoid(f, a, b, n):
    """Composite trapezoid rule. Order 2."""
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def composite_simpson(f, a, b, n):
    """Composite Simpson rule. Requires an even number of panels. Order 4.

    One Simpson application per pair of panels, so the interior seams get
    their weights from being shared: 4 on every pair midpoint, 1 + 1 = 2 on
    every seam, which is the classic 1, 4, 2, 4, ..., 4, 1 pattern.
    """
    if n % 2 == 1:
        n += 1                              # Simpson pairs panels; force even
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    total = 0.0
    for k in range(n // 2):
        total = total + (h / 3.0) * (y[2*k] + 4.0 * y[2*k + 1] + y[2*k + 2])
    return total

In [ ]:
# ---------------------------------------------------------------------------
# Measure the composite convergence rate
# ---------------------------------------------------------------------------
# Test integral: exp(x) on [0, 1], whose exact value is e - 1. We halve the
# panel width repeatedly and fit a line to log(error) vs log(n). The fitted
# slope is the empirical order of the rule.
import math

def f_smooth(x): return np.exp(x)
A, B = 0.0, 1.0
I_EXACT = math.e - 1.0

def show_convergence():
    """Plot composite error vs number of panels and report the fitted order."""
    ns = 2 ** np.arange(1, 9)               # 2, 4, ..., 256 panels
    rules = {
        "trapezoid": composite_trapezoid,
        "midpoint":  composite_midpoint,
        "Simpson":   composite_simpson,
    }
    plt.figure()
    for name, rule in rules.items():
        err = np.array([abs(rule(f_smooth, A, B, n) - I_EXACT) for n in ns])
        slope = np.polyfit(np.log(ns), np.log(err + 1e-18), 1)[0]
        plt.loglog(ns, err + 1e-18, ".-", label=f"{name}: slope {slope:.2f}")
    plt.xlabel("number of panels n"); plt.ylabel("absolute error")
    plt.title("Composite quadrature error (integrand exp x on [0,1])")
    plt.legend(); plt.show()
    print("trapezoid and midpoint converge at order 2, Simpson at order 4")

show_convergence()

In [ ]:
# ---------------------------------------------------------------------------
# Degree of exactness: Simpson integrates cubics exactly
# ---------------------------------------------------------------------------
# A rule has degree of exactness m if it integrates every polynomial of degree
# up to m with zero error. Simpson's is 3, one higher than its interpolation
# degree, thanks to a symmetry cancellation.
def check_exactness():
    """Integrate monomials x^k on [0,1] with a single Simpson panel (n=2)."""
    print("Simpson on [0,1], single application (n = 2 panels):")
    for k in range(0, 5):
        g = lambda x, k=k: x ** k
        approx = composite_simpson(g, 0.0, 1.0, 2)
        exact  = 1.0 / (k + 1)
        print(f"  x^{k}: error = {abs(approx - exact):.2e}")

check_exactness()

In [ ]:
# ---------------------------------------------------------------------------
# Adaptive Simpson with an interval-halving error estimate
# ---------------------------------------------------------------------------
# On each subinterval, compare one Simpson estimate for the whole of it
# against the sum of two half-interval estimates. By Richardson reasoning
# their difference divided by 15 estimates the error of the finer value; if
# that is below the tolerance the value is accepted, with the estimate added
# on as a cheap accuracy boost, and otherwise both halves recurse. A depth
# cap prevents runaway subdivision.

def simpson_panel(f, a, b):
    """Single Simpson estimate on [a, b] using endpoints and midpoint."""
    c = (a + b) / 2.0
    return (b - a) / 6.0 * (f(a) + 4.0 * f(c) + f(b))

def adaptive_simpson(f, a, b, tol, whole, depth):
    """Recursive core. whole is the single-panel estimate for [a, b]."""
    c = (a + b) / 2.0
    left = simpson_panel(f, a, c)
    right = simpson_panel(f, c, b)
    if depth >= 50 or abs(left + right - whole) <= 15.0 * tol:
        return left + right + (left + right - whole) / 15.0
    return (adaptive_simpson(f, a, c, tol, left, depth + 1) +
            adaptive_simpson(f, c, b, tol, right, depth + 1))

In [ ]:
# ---------------------------------------------------------------------------
# When does adaptivity pay for itself?
# ---------------------------------------------------------------------------
# Adaptive Simpson costs more per unit of interval than a uniform rule: it
# re-evaluates f at the same points as it recurses, and it spends evaluations
# on the error estimate itself. It wins only when a uniform grid would be
# forced to stay fine everywhere. Two integrands, opposite verdicts.

def count_adaptive(f, a, b, tol):
    """Run adaptive Simpson, returning (value, distinct evaluation points)."""
    seen = []
    def recorded(x):
        seen.append(round(float(x), 14))
        return f(x)
    val = adaptive_simpson(recorded, a, b, tol, simpson_panel(recorded, a, b), 0)
    return val, sorted(set(seen))

def uniform_cost(f, a, b, exact, target_err):
    """Smallest power-of-two panel count whose composite Simpson error fits."""
    n = 2
    while abs(composite_simpson(f, a, b, n) - exact) > target_err and n < 2 ** 22:
        n *= 2
    return n

def compare(f, a, b, exact, tol):
    val, pts = count_adaptive(f, a, b, tol)
    err = abs(val - exact)
    n = uniform_cost(f, a, b, exact, err) + 1
    print(f"  adaptive: {len(pts):6d} evaluations, error {err:.2e}")
    if n >= len(pts):
        print(f"  uniform : {n:6d} evaluations for the same error, "
              f"{n / len(pts):.0f} times the adaptive cost")
    else:
        print(f"  uniform : {n:6d} evaluations for the same error, "
              f"so adaptive costs {len(pts) / n:.1f} times more")
    return pts

def f_bump(x): return np.exp(-200.0 * (x - 0.5) ** 2)
I_BUMP = 0.5 * np.sqrt(np.pi / 200.0) * (
    math.erf(np.sqrt(200.0) * 0.5) - math.erf(-np.sqrt(200.0) * 0.5))

print("A narrow but perfectly smooth bump, exp(-200(x-1/2)^2) on [0,1]:")
bump_pts = compare(f_bump, 0.0, 1.0, I_BUMP, 1e-8)
print("  A Gaussian has bounded derivatives of every order, so the uniform")
print("  rule's h^4 is in no trouble and the extra machinery does not pay.")
print()
print("sqrt(x) on [0,1], whose second derivative blows up at the origin:")
sqrt_pts = compare(np.sqrt, 0.0, 1.0, 2.0 / 3.0, 1e-8)
print("  The uniform rule's error is governed by the largest fourth derivative")
print("  anywhere on the interval, which is unbounded here, so it has to stay")
print("  fine everywhere. Adaptive refinement pays for the bad end only.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, pts, f, title in [
        (axes[0], bump_pts, f_bump, f"smooth bump: {len(bump_pts)} evaluations"),
        (axes[1], sqrt_pts, np.sqrt, f"sqrt(x): {len(sqrt_pts)} evaluations")]:
    xs = np.linspace(0, 1, 800)
    ax.plot(xs, f(xs), "b-", lw=2)
    ax.plot(pts, np.full(len(pts), -0.05), "r|", ms=8)
    ax.set_title(title); ax.set_xlabel("x")
plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# The estimate is only as good as the samples it is built from
# ---------------------------------------------------------------------------
# Put the same bump on a wider interval and run the method again. Nothing
# about the integrand has changed; only the interval has.

for right_end in [1.0, 2.0, 4.0, 8.0]:
    exact = 0.5 * np.sqrt(np.pi / 200.0) * (
        math.erf(np.sqrt(200.0) * (right_end - 0.5))
        + math.erf(np.sqrt(200.0) * 0.5))
    val, pts = count_adaptive(f_bump, 0.0, right_end, 1e-8)
    first = ", ".join(f"{t:g}" for t in pts[:3])
    print(f"[0, {right_end:g}]: exact {exact:.6f}   computed {val:.6f}   "
          f"error {abs(val - exact):.1e}   {len(pts):4d} evaluations   "
          f"first samples at {first}")

print()
print("On [0, 4] the first three samples land at 0, 2 and 4, where the bump is")
print("indistinguishable from zero. The whole-interval estimate is 0, both half")
print("estimates are 0, their difference is 0, and the tolerance test passes on")
print("the first try. The method reports 0 and stops, confidently and cheaply.")

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: how the verdict moves with the tolerance
# ---------------------------------------------------------------------------
# Tighten the tolerance and watch the two columns separate. The adaptive cost
# grows slowly, the uniform cost grows like tol^(-1/4) on a smooth integrand
# and far faster once smoothness fails at a point.

def show_costs(log10_tol=-8.0):
    tol = 10.0 ** log10_tol
    print(f"tolerance {tol:.0e}")
    print(f"{'integrand':>22}  {'adaptive':>9}  {'uniform':>9}  {'ratio':>8}")
    for name, f, a, b, exact in [
            ("exp(-200(x-1/2)^2)", f_bump, 0.0, 1.0, I_BUMP),
            ("sqrt(x)", np.sqrt, 0.0, 1.0, 2.0 / 3.0),
            ("x^(1/3)", np.cbrt, 0.0, 1.0, 3.0 / 4.0)]:
        val, pts = count_adaptive(f, a, b, tol)
        err = abs(val - exact)
        n = uniform_cost(f, a, b, exact, err) + 1
        ratio = n / len(pts)
        verdict = f"{ratio:7.0f}x" if ratio >= 1 else f"  1/{1/ratio:.1f}x"
        print(f"{name:>22}  {len(pts):9d}  {n:9d}  {verdict}")

interact(show_costs,
         log10_tol=FloatSlider(value=-8.0, min=-12.0, max=-3.0, step=0.5,
                               description="log10(tol)"));

## Summary

- Newton-Cotes rules integrate the interpolant on equispaced nodes; composite versions sum the rule over many panels.
- Composite trapezoid and midpoint converge at order 2, composite Simpson at order 4; the log-log error slope recovers these orders.
- Simpson has degree of exactness 3, one above its interpolation degree, so it is exact for cubics.
- Adaptive Simpson compares one panel estimate against two half-panel estimates and subdivides where they disagree. It is not free: it re-evaluates $f$ as it recurses, so on an integrand that is merely peaked, like a Gaussian bump, it costs more than a uniform rule of the same accuracy.
- What it wins on is a *local* loss of smoothness. A uniform rule's error is governed by the largest fourth derivative anywhere on the interval, so one bad point forces a fine grid everywhere, while adaptive refinement pays for the bad point only. On $\sqrt{x}$ over $[0,1]$ the saving is a factor of several hundred, and it grows as the tolerance tightens.
- The error estimate sees only the points it samples. A feature falling between the first few samples is invisible, the tolerance test passes immediately, and the method returns a confident wrong answer.